In [3]:
ADLSaccount = "onelake"
containerName = "Fabmigration"
sourceFolder = "Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2"
targetFolder = "Synapsemigration.Lakehouse/Files/silver/"
targetEnv = "silver"
entityName = "Sales.CurrencyRate"
fileExtension = "parquet"

StatementMeta(, eb4f3b02-f643-4e28-af57-2acc449a4350, 5, Finished, Available, Finished, False)

In [2]:
%run /00_Functions

StatementMeta(, eb4f3b02-f643-4e28-af57-2acc449a4350, 4, Finished, Available, Finished, True)

In [4]:
# Set file path to load to DF
sourcePath = f'abfss://{containerName}@{ADLSaccount}.dfs.fabric.microsoft.com/{sourceFolder}/{entityName}.{fileExtension}'
targetPath = f'abfss://{containerName}@{ADLSaccount}.dfs.fabric.microsoft.com/{targetFolder}/'

StatementMeta(, eb4f3b02-f643-4e28-af57-2acc449a4350, 6, Finished, Available, Finished, False)

In [5]:
# Read parquet file to get data
dfdata = spark.read.load(sourcePath, format='parquet')

# Clean
dfdata = clean_df(dfdata)

# Set the schema for entity. Also if needed the first row.
entitySchema = GetSaleTableSchema(entityName)
firstRow = GetSaleTableFirstRow(entityName, dfdata)

# Create empty DF with schema and first row (if needed)
df = spark.createDataFrame(firstRow, entitySchema)

# Update column names from parquet file if needed
dfdata = UpdateDFColNames(entityName, dfdata)

# Join empty Df with data from parquet
df = df.unionByName(dfdata)

#display(df.limit(10))

StatementMeta(, eb4f3b02-f643-4e28-af57-2acc449a4350, 7, Finished, Available, Finished, False)

In [20]:
# Set name for silver
entityNameFix = f'silver_{entityName.replace(".", "_")}'

# Save DF into stage after clean process
SaveDF(df, targetEnv, entityNameFix, targetPath)

StatementMeta(fabmigspark, 73, 25, Finished, Available, Finished)